In [2]:
import polars as pl
from sentence_transformers import SentenceTransformer
import numpy as np
import json
from google.colab import drive

EMBEDDING_MODEL = "ibm-granite/granite-embedding-30m-english"

In [3]:
drive.mount('/content/drive') # Mount Google Drive to access data files

Mounted at /content/drive


In [4]:
# Load filtered book parquet from Google Drive
books_df = pl.read_parquet('/content/drive/MyDrive/books_with_genres.parquet')

In [8]:
print(len(books_df))

224


In [7]:
# Load embedding model
model = SentenceTransformer(EMBEDDING_MODEL, device = "cuda")

# Generate embeddings for book descriptions + title concatenated, do this in batches to avoid memory issues
BATCH_SIZE = 512 # Adjust batch size based on available memory
embeddings = [] # Store embeddings in a list to avoid memory issues

# Create rows of books with concatenated title and description - handle nulls with list comprehension
books_df = books_df.with_columns(
    pl.when(pl.col("description").is_null() | (pl.col("description")== ""))
    .then(pl.col("title"))
    .otherwise(pl.col("title") + " - " + pl.col("description"))
    .alias("text")
)

# Call model.encode on this with batch param
embeddings = model.encode(
    books_df["text"].to_list(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)


# Save matrix of embeddings as .npy file back to drive
np.save('/content/drive/MyDrive/book_embeddings.npy', np.array(embeddings))

# Save book_id to row index mapping as a json file to drive, this will be used later to map from book_id to embedding index when we want to retrieve embeddings for a given book_id
book_id_to_index = {row["book_id"]: idx for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open('/content/drive/MyDrive/book_id_to_index.json', 'w') as file:
    json.dump(book_id_to_index, file, indent=2)

# Build the reverse mapping from row index to book_id, this will be used later to map from embedding index back to book_id when we want to retrieve metadata for a given embedding
index_to_book_id = {idx: row["book_id"] for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open('/content/drive/MyDrive/index_to_book_id.json', 'w') as file:
    json.dump(index_to_book_id, file, indent=2)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]